# **Modeling an Intelligent Vacuum Agent Using Alternative MAS Libraries**

#### Emilio Alejandro González Huerta A01286440
---
## Formal Modelling

### Agent Model

Its **internal state** consists of two things: 

- its current position on the grid, stored as a `(row, col)` tuple in `self.pos`
- and a counter of how many cells it has cleaned so far (`self.cleaned_count`). It does not remember past positions or plan ahead.

**Perception**: Each agent has full observability. Each step it scans `dirtgrid` and gets a list of every cell with value `1`, so it always knows exactly where all the dirt is.

**Actions**: Each step the agent does exactly one of the following:
- **Clean**: if standing on a dirty cell, sets it to `0` and increments the counter.
- **Move**: walks one step toward the nearest dirty cell (up, down, left, or right, not diagonal).

Avoids colliding with other robots (moving towards same cell)

- **Stay**: does nothing if the grid is fully clean.

--> Cleaning always takes priority over moving.

---

### Environment Model

**Grid representation**: A 10×10 NumPy array (`self.dirtgrid`) where `0` means clean and `1` means dirty.

**Dirt generation**: Dirt is introduced at setup, `initial_dirty` random cells are set to `1`. Then during the run, every `dirt_interval` steps one random cell is dirtied again, even if it was already dirty.

**Determinism**: Each agent's behavior is fully deterministic, as the decision doesn't vary with the same grid.

The environment itself is stochastic because dirt spawns at random positions.

---

### State Transition Function

Every step runs in this exact order:

1. If `step % dirt_interval == 0`, one random cell becomes dirty.
2. Every agent checks its current cell. If dirty, it cleans it.
3. Otherwise, each agent finds the nearest dirty cell by Manhattan distance (to pick the target), then runs a BFS from its current position to that target and takes only the first step of the resulting path. BFS implementation may be inneficient, nonetheless the agent can be scaled to having obstacles in its grid, where only using manhattan distance would be insufficient.

---

### Data Structures

Two classes drive the simulation: 
- `RoomModel(mesa.Model)` manages the grid and the loop &
- `VacuumAgent(mesa.Agent)` handles perception and movement.

| Concept | Implementation |
|---|---|
| Grid state | `np.ndarray (N, N)` :integers `0` or `1` |
| Agent position | `self.pos` :`(row, col)` tuple on each agent |
| Dirt positions | No separate structure, scanned from `dirtgrid` |
| Grid snapshots | `model.grid_history` :list of grid copies one per step |
| Position snapshots | `model.pos_history` :list of `(row, col)` lists, one per agent per step |

---

## Implementation

#### Libraries

In [ ]:
import mesa
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from collections import deque

#### Configuration Variables
Contains global important variables to the agent.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
GRID_SIZE      = 25   # room size (N x N)
INITIAL_DIRTY  = 140   # dirty cells at the start
DIRT_INTERVAL  = 3    # steps between each new dirty cell
NUM_AGENTS     = 4    # number of vacuum robots
STEPS          = 200  # simulation length
SEED           = 676767
# ─────────────────────────────────────────────────────────────────────────────

BFS Implementtation

In [ ]:
def bfs_next_step(start, goal, N, blocked):
    # returns the first step of the shortest path from start to goal,
    # avoiding any cell in the blocked set (other agents' positions/targets)
    if start == goal:
        return start
    visited = {start}
    queue = deque([(start, [])])
    while queue:
        (r, c), path = queue.popleft()
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nb = (r+dr, c+dc)
            if (0 <= nb[0] < N and 0 <= nb[1] < N
                    and nb not in visited and nb not in blocked):
                new_path = path + [nb]
                if nb == goal:
                    return new_path[0]
                visited.add(nb)
                queue.append((nb, new_path))
    return start  # stay put if no path found

### Class

In [ ]:
class VacuumAgent(mesa.Agent):

    def __init__(self, model):
        super().__init__(model)
        self.pos           = None
        self.cleaned_count = 0
        self.target        = None  # dirty cell this agent is heading toward

    def nearest_dirty(self, claimed):
        # pick the closest dirty cell that no other agent has already claimed
        N = self.model.N
        dirty = [(r,c) for r in range(N) for c in range(N)
                 if self.model.dirtgrid[r,c] == 1 and (r,c) not in claimed]
        if not dirty:
            return None
        r0, c0 = self.pos
        return min(dirty, key=lambda rc: abs(rc[0]-r0) + abs(rc[1]-c0))

    def step(self, occupied, claimed):
        # occupied : set of cells where other agents currently stand
        # claimed  : set of dirty cells already targeted by other agents
        r, c = self.pos

        if self.model.dirtgrid[r, c] == 1:
            # clean current cell if dirty
            self.model.dirtgrid[r, c] = 0
            self.cleaned_count += 1
            self.target = None
        else:
            # find a target that no other agent is already heading to
            self.target = self.nearest_dirty(claimed)
            if self.target:
                # move via BFS, avoiding cells occupied by other agents
                next_pos = bfs_next_step(self.pos, self.target, self.model.N, occupied)
                # only move if the next cell is not occupied
                if next_pos not in occupied:
                    self.pos = next_pos


class RoomModel(mesa.Model):

    def __init__(self, N, initial_dirty, dirt_interval, num_agents, seed):
        super().__init__(rng=seed)
        self.N             = N
        self.dirt_interval = dirt_interval
        self.t             = 0

        # initialize clean grid and place dirty cells randomly
        self.dirtgrid = np.zeros((N, N), dtype=int)
        cells = [(r,c) for r in range(N) for c in range(N)]
        for r,c in self.random.sample(cells, initial_dirty):
            self.dirtgrid[r, c] = 1

        # create agents and place each one at a unique random position
        start_positions = self.random.sample(cells, num_agents)
        self.vacuums = []
        for pos in start_positions:
            agent = VacuumAgent(self)
            agent.pos = pos
            self.vacuums.append(agent)

        # snapshots per step for the animation
        self.grid_history = []
        self.pos_history  = []

    def step(self):
        self.t += 1

        # spawn a new dirty cell every dirt_interval steps
        if self.t % self.dirt_interval == 0:
            r = self.random.randrange(self.N)
            c = self.random.randrange(self.N)
            self.dirtgrid[r, c] = 1

        # activate agents one by one, updating occupied/claimed after each move
        occupied = set()
        claimed  = set()

        for agent in self.vacuums:
            # build blocked set from all OTHER agents' current positions
            other_positions = {a.pos for a in self.vacuums if a is not agent}
            other_targets   = {a.target for a in self.vacuums
                               if a is not agent and a.target is not None}

            agent.step(other_positions, other_targets)

            # register this agent's new position and target so the next
            # agent in line treats them as taken
            occupied.add(agent.pos)
            if agent.target:
                claimed.add(agent.target)

        self.grid_history.append(self.dirtgrid.copy())
        self.pos_history.append([agent.pos for agent in self.vacuums])

### Simulation

In [ ]:
# run the full simulation before animating
model = RoomModel(GRID_SIZE, INITIAL_DIRTY, DIRT_INTERVAL, NUM_AGENTS, SEED)
for _ in range(STEPS):
    model.step()

total_cleaned = sum(a.cleaned_count for a in model.vacuums)

# colors for each agent marker
AGENT_COLORS = ['red', 'blue', 'green', 'orange', 'purple',
                'cyan', 'magenta', 'yellow', 'lime', 'pink']

fig, ax = plt.subplots(figsize=(5, 5))

def animate(frame):
    ax.clear()
    ax.imshow(model.grid_history[frame], cmap='Greys', vmin=0, vmax=1, origin='upper')

    # draw each agent with its own color
    for i, pos in enumerate(model.pos_history[frame]):
        r, c = pos
        color = AGENT_COLORS[i % len(AGENT_COLORS)]
        ax.plot(c, r, 'o', color=color, markersize=14)

    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(
        f"Step {frame+1}  |  "
        f"Cleaned: {total_cleaned}  |  "
        f"Dirty: {int(model.grid_history[frame].sum())}"
    )

anim = animation.FuncAnimation(fig, animate, frames=STEPS, interval=150, repeat=False)
plt.close()

HTML(anim.to_jshtml())

---
### Comparative Analysis: Mesa vs AgentPy

The library used for this simulation was ``Mesa 3.x``.

Mesa is more explicit than AgentPy in OOP principles, as you define `Model` and `Agent` as separate classes with clear responsibilities, which makes the architecture easier to understand and extend. AgentPy is more compact and better suited for prototypes, but Mesa's structure scales better as the model grows in complexity. Mesa handles multi-agent systems more naturally. Adding `NUM_AGENTS` robots, collision avoidance, and target claiming required minimal restructuring. In AgentPy, replicating the same coordination logic would have required more workarounds since it lacks native per-agent state management at the same level.

Mesa gives more fine control. Analogous to C++ to Java. Managing the loop with bfs + locked cells and claimed targets per step was straightforward because it's fully manual. AgentPy abstracts some of that, which is convenient unless in specific cases. For large-scale systems Mesa would be the preferred choice. The explicit class structure, flexible scheduling, and stronger community support make it easier to maintain and extend as complexity grows. AgentPy is a good fit for smaller, faster experiments, but it starts to show its limits when agent coordination and custom step logic are involved.